In [ ]:
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_ENDPOINT = os.getenv("LANGSMITH_ENDPOINT")

In [ ]:
import bs4
from langchain_classic import hub
from langsmith import Client
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_openai import OpenAIEmbeddings

## Indexing

In [ ]:
# load documents
loader = WebBaseLoader(
    web_path=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

In [ ]:
# split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

In [ ]:
# embed
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2", api_key=GEMINI_API_KEY)

vectorstore = Chroma.from_documents(documents=splits,
                                    embedding=embeddings)

In [ ]:
retriever = vectorstore.as_retriever()

## Retrieval and Generation 

In [ ]:
client = Client(
    api_key=LANGSMITH_API_KEY,
    api_url=LANGSMITH_ENDPOINT
)

In [ ]:
# prompt --- won't work in the EU, this one is for the USA only
# prompt = client.pull_prompt("rlm/rag-prompt", dangerously_pull_public_prompt=True)

In [ ]:
import json
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
prompt = ChatPromptTemplate.from_template(
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know. "
    "Use three sentences maximum and keep the answer concise.\n"
    "Question: {question} \nContext: {context} \nAnswer:"
)

In [ ]:
# LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0,
    api_key=GEMINI_API_KEY
)

In [ ]:
# post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
# chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# question
rag_chain.invoke("What is Task Decomposition?")

## Indexing under the hood

In [ ]:
# documents
question = "What kind of pets do I like?"
document = "My favourite pet is a cat."

In [ ]:
import tiktoken

In [ ]:
# count tokens considering ~4 char / token
def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

In [ ]:
num_tokens_from_string(question, "cl100k_base")

In [ ]:
# text embedding models
embd = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2", api_key=GEMINI_API_KEY)
query_result = embd.embed_query(question)
document_result = embd.embed_query(document)
len(query_result)

### Cosine similarity

In [ ]:
import numpy as np

In [ ]:
def cosine_similarity(vec1, vec2):
    norm_vec1 = vec1 / np.linalg.norm(vec1)
    norm_vec2 = vec2 / np.linalg.norm(vec2)

    return norm_vec1@norm_vec2

In [ ]:
similarity = cosine_similarity(query_result, document_result)
print("Cosine Similarity:", similarity)

### Document loaders

In [ ]:
### indexing ###
loader = WebBaseLoader(
    web_path=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

`RecursiceCharacterTextSplitter` splits the simple text into paragraphs first, then sentences, then words. It tries to split on them in order until the chunks are small enough. It's trying to keep all paragraphs (and then sentences, and then words) together as long as possible, as those would generally seem to be the strongest semantically related pieces of text.

In [ ]:
# split
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50
)

# make splits
splits = text_splitter.split_documents(blog_docs)

In [ ]:
# index
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2", api_key=GEMINI_API_KEY)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings
)

retriever = vectorstore.as_retriever()

## Retrieval under the hood

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 1}) # number of documents to search

In [ ]:
docs = retriever._get_relevant_documents("What is Task Decomposition?", run_manager="")
print(docs[0])
print(len(docs))

## Generation under the hood

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate 

In [ ]:
# prompt
template = """Answer the question based on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
prompt

In [ ]:
# LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0,
    api_key=GEMINI_API_KEY
)

In [ ]:
# LangChain expression language (LCEL)
# lets easily compose prompts, LLMs, parsers, retrievers and other things
# in the example below I'm connecting a prompt to an llm
chain = prompt | llm

In [ ]:
# invoking the chain with a dict
chain.invoke(
    {
        "context":blog_docs,
        "question":"What is Task Decomposition?"
    }
)

In [ ]:
# rag chain that is going to retrieve the relevant information automatically
# it automatically triggers the retriever which will return documents
# which get passed into the context
rag_chain = (
    {
        "context": retriever, # invokes retriever.invoke(question)
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

`retriever.invoke(question)` is a public analog of `retriever._get_relevant_documents(question)`

use `format_docs()` to merge the `Document` list into a text, so there's no technical metadata of `Document(...)` objects:

```python
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
```

and then:

```python
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    ...
)
```

In [ ]:
rag_chain.invoke("What is Task Decomposition?")